# Session Explorer

Load and visualize a sensor capture session from the Sensor Capture app.

**Sensors captured**: Accelerometer (~500 Hz), Gyroscope (~500 Hz), Magnetometer (~50 Hz), GPS (~1 Hz)

If no real data is available, a synthetic session with simulated pothole events is generated automatically.

**Works in Google Colab** — no local install needed. Open via:
```
https://colab.research.google.com/github/SebastienBinet/NidsDePoule/blob/data_analysis_tools/data_capture_tools/analysis/notebooks/01_explore_session.ipynb
```

In [ ]:
# Install dependencies (needed for Colab; harmless if already installed locally)
%pip install -q folium scipy pandas numpy matplotlib

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import folium

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100


# --- Inline session loader (works in Colab without local package install) ---

def load_session(session_dir):
    """Load all CSV files and metadata from a capture session directory."""
    d = Path(session_dir)
    meta_files = list(d.glob("meta_*.json"))
    if not meta_files:
        raise FileNotFoundError(f"No meta_*.json found in {d}")
    meta = json.loads(meta_files[0].read_text())

    start_boot_ns = meta.get("start_boot_time_ns", 0)
    boot_offset_ms = meta.get("boot_to_epoch_offset_ms", 0)

    def _load_sensor(prefix):
        files = list(d.glob(f"{prefix}_*.csv"))
        if not files:
            return None
        df = pd.read_csv(files[0])
        if "timestamp_ns" in df.columns:
            t0 = start_boot_ns if start_boot_ns > 0 else df["timestamp_ns"].iloc[0]
            df["time_s"] = (df["timestamp_ns"] - t0) / 1e9
        return df

    accel = _load_sensor("accel")
    gyro = _load_sensor("gyro")
    mag = _load_sensor("mag")

    gps_files = list(d.glob("gps_*.csv"))
    gps = None
    if gps_files:
        gps = pd.read_csv(gps_files[0])
        if "timestamp_ms" in gps.columns:
            start_epoch_ms = boot_offset_ms + start_boot_ns / 1e6
            gps["time_s"] = (gps["timestamp_ms"] - start_epoch_ms) / 1e3

    # Load events (ground-truth labels from BT button, screen tap, etc.)
    events_files = list(d.glob("events_*.csv"))
    events = None
    if events_files:
        events = pd.read_csv(events_files[0])
        if "timestamp_ms" in events.columns:
            start_epoch_ms = boot_offset_ms + start_boot_ns / 1e6
            events["time_s"] = (events["timestamp_ms"] - start_epoch_ms) / 1e3

    return {"meta": meta, "accel": accel, "gyro": gyro, "mag": mag, "gps": gps, "events": events}


def plot_event_markers(ax, events, t_start=None, t_end=None):
    """Draw vertical lines for labeled events on a matplotlib axis."""
    if events is None or len(events) == 0:
        return
    colors = {"pothole": "red", "crack": "orange", "rough": "brown", "other": "gray"}
    for _, ev in events.iterrows():
        t = ev["time_s"]
        if t_start is not None and (t < t_start or t > t_end):
            continue
        c = colors.get(ev.get("event_type", "other"), "gray")
        ax.axvline(t, color=c, linestyle="--", alpha=0.7, linewidth=1.2)


print('Ready.')

## Load session (or generate synthetic data)

In [ ]:
# --- Set SESSION_DIR to a real session path, or leave None for synthetic data ---
SESSION_DIR = None  # e.g. '../../sample_data/session_20260404_143025_abc12'


def generate_synthetic_session(duration_s=120, accel_hz=500, gyro_hz=500, gps_hz=1):
    """Generate a fake session with 3 simulated pothole events."""
    rng = np.random.default_rng(42)
    start_boot_ns = 1_000_000_000_000  # 1000 seconds after boot

    # --- Accelerometer ---
    n_accel = duration_s * accel_hz
    t_accel_ns = start_boot_ns + np.arange(n_accel) * (1e9 / accel_hz)
    t_accel_ns = t_accel_ns.astype(np.int64)

    # Baseline: gravity on Z (~9.81) + small noise on all axes
    ax = rng.normal(0, 0.15, n_accel).astype(np.float32)
    ay = rng.normal(0, 0.15, n_accel).astype(np.float32)
    az = rng.normal(9.81, 0.20, n_accel).astype(np.float32)

    # Inject 3 pothole events at t=30s, 65s, 95s
    pothole_times = [30.0, 65.0, 95.0]
    for t_event in pothole_times:
        idx = int(t_event * accel_hz)
        width = int(0.08 * accel_hz)  # ~80ms impulse
        # Gaussian envelope pothole impulse
        envelope = np.exp(-0.5 * ((np.arange(width * 4) - width * 2) / (width * 0.5)) ** 2)
        peak = rng.uniform(8, 25)  # peak magnitude in m/s^2
        impulse = envelope * peak * np.sin(2 * np.pi * 12 * np.arange(len(envelope)) / accel_hz)
        end = min(idx + len(impulse), n_accel)
        az[idx:end] += impulse[: end - idx].astype(np.float32)
        ax[idx:end] += (impulse[: end - idx] * 0.3 * rng.choice([-1, 1])).astype(np.float32)

    accel = pd.DataFrame({
        'timestamp_ns': t_accel_ns, 'x_ms2': ax, 'y_ms2': ay, 'z_ms2': az,
    })
    accel['time_s'] = (accel['timestamp_ns'] - start_boot_ns) / 1e9

    # --- Gyroscope ---
    n_gyro = duration_s * gyro_hz
    t_gyro_ns = start_boot_ns + np.arange(n_gyro) * (1e9 / gyro_hz)
    gx = rng.normal(0, 0.01, n_gyro).astype(np.float32)
    gy = rng.normal(0, 0.01, n_gyro).astype(np.float32)
    gz = rng.normal(0, 0.005, n_gyro).astype(np.float32)
    # Gyro responds to potholes too (pitch impulse)
    for t_event in pothole_times:
        idx = int(t_event * gyro_hz)
        width = int(0.1 * gyro_hz)
        envelope = np.exp(-0.5 * ((np.arange(width * 4) - width * 2) / (width * 0.5)) ** 2)
        gy[idx:idx + len(envelope)] += (envelope * rng.uniform(0.5, 2.0)).astype(np.float32)

    gyro = pd.DataFrame({
        'timestamp_ns': t_gyro_ns.astype(np.int64), 'x_rads': gx, 'y_rads': gy, 'z_rads': gz,
    })
    gyro['time_s'] = (gyro['timestamp_ns'] - start_boot_ns) / 1e9

    # --- GPS (simulating a drive in Montreal) ---
    n_gps = duration_s * gps_hz
    lat_start, lon_start = 45.5088, -73.5878
    speed = 12.0  # ~43 km/h
    bearing = 45.0  # northeast
    boot_offset_ms = 1_712_000_000_000  # arbitrary epoch offset
    t_gps_ms = (boot_offset_ms + (start_boot_ns / 1e6)
                + np.arange(n_gps) * (1000 / gps_hz)).astype(np.int64)

    # Simple straight-line drive
    dlat = speed * np.cos(np.radians(bearing)) / 111_320  # deg per second
    dlon = speed * np.sin(np.radians(bearing)) / (111_320 * np.cos(np.radians(lat_start)))
    lats = lat_start + dlat * np.arange(n_gps)
    lons = lon_start + dlon * np.arange(n_gps)

    gps = pd.DataFrame({
        'timestamp_ms': t_gps_ms,
        'lat_deg': lats, 'lon_deg': lons,
        'altitude_m': 45.0, 'speed_mps': speed + rng.normal(0, 0.5, n_gps),
        'bearing_deg': bearing + rng.normal(0, 2, n_gps),
        'accuracy_m': 3.0 + rng.exponential(1, n_gps),
        'vertical_accuracy_m': 5.0,
        'speed_accuracy_mps': 0.5,
        'bearing_accuracy_deg': 5.0,
    })
    gps['time_s'] = np.arange(n_gps, dtype=float)

    # --- Events (simulating BT button presses ~1s after each pothole) ---
    events = pd.DataFrame({
        'timestamp_ms': [int(boot_offset_ms + (start_boot_ns / 1e6) + (t + 1.0) * 1000)
                         for t in pothole_times],
        'event_type': ['pothole', 'pothole', 'pothole'],
        'source': ['bt_button', 'bt_button', 'screen_button'],
    })
    events['time_s'] = [t + 1.0 for t in pothole_times]  # ~1s reaction delay

    meta = {
        'session_id': 'synthetic_demo',
        'device_model': 'Synthetic',
        'android_version': 'N/A',
        'start_time_iso': '2026-04-04T14:30:00.000Z',
        'start_time_epoch_ms': int(boot_offset_ms + start_boot_ns / 1e6),
        'start_boot_time_ns': int(start_boot_ns),
        'boot_to_epoch_offset_ms': int(boot_offset_ms),
        'sample_counts': {
            'accel': int(n_accel), 'gyro': int(n_gyro), 'mag': 0, 'gps': int(n_gps),
            'events': len(events),
        },
    }

    return {'meta': meta, 'accel': accel, 'gyro': gyro, 'mag': None, 'gps': gps, 'events': events}


# Load real or synthetic data
if SESSION_DIR and Path(SESSION_DIR).exists():
    session = load_session(SESSION_DIR)
    print(f"Loaded real session: {SESSION_DIR}")
else:
    session = generate_synthetic_session()
    print('Using synthetic data (set SESSION_DIR to load a real session)')

meta = session['meta']
accel = session['accel']
gyro = session['gyro']
mag = session['mag']
gps = session['gps']
events = session['events']

if events is not None and len(events) > 0:
    print(f"\nLabeled events: {len(events)}")
    print(events.to_string(index=False))

## Session metadata

In [ ]:
counts = meta.get('sample_counts', {})
duration_s = accel['time_s'].iloc[-1] - accel['time_s'].iloc[0] if accel is not None else 0

print(f"Session:   {meta.get('session_id', '?')}")
print(f"Device:    {meta.get('device_model', '?')}")
print(f"Android:   {meta.get('android_version', '?')}")
print(f"Start:     {meta.get('start_time_iso', '?')}")
print(f"Duration:  {duration_s:.1f} s ({duration_s/60:.1f} min)")
print()
print('Sample counts and actual rates:')
for sensor, count in counts.items():
    hz = count / duration_s if duration_s > 0 else 0
    print(f"  {sensor:>8s}: {count:>10,} samples  ({hz:>7.1f} Hz)")

## Accelerometer — 3-axis time series

**Raw** (with gravity): X, Y, Z as reported by `TYPE_ACCELEROMETER`. At rest, Z reads ~9.81 m/s².

**Gravity removed** (linear acceleration): gravity estimated via low-pass filter and subtracted. At rest, all axes read ~0. The magnitude plot uses these gravity-removed values — it shows only the dynamic acceleration (bumps, potholes, braking).

In [ ]:
if accel is not None and len(accel) > 0:
    # Downsample for overview plot (every Nth sample)
    step = max(1, len(accel) // 10_000)
    a = accel.iloc[::step].copy()

    # Estimate gravity via low-pass filter (cutoff ~0.5 Hz)
    alpha = 0.01  # low-pass coefficient: smaller = smoother gravity estimate
    grav_x = np.zeros(len(accel), dtype=np.float32)
    grav_y = np.zeros(len(accel), dtype=np.float32)
    grav_z = np.zeros(len(accel), dtype=np.float32)
    grav_x[0] = accel['x_ms2'].iloc[0]
    grav_y[0] = accel['y_ms2'].iloc[0]
    grav_z[0] = accel['z_ms2'].iloc[0]
    for i in range(1, len(accel)):
        grav_x[i] = alpha * accel['x_ms2'].iloc[i] + (1 - alpha) * grav_x[i - 1]
        grav_y[i] = alpha * accel['y_ms2'].iloc[i] + (1 - alpha) * grav_y[i - 1]
        grav_z[i] = alpha * accel['z_ms2'].iloc[i] + (1 - alpha) * grav_z[i - 1]

    # Compute linear acceleration (gravity removed) on full data, then downsample
    accel_lin_x = accel['x_ms2'].values - grav_x
    accel_lin_y = accel['y_ms2'].values - grav_y
    accel_lin_z = accel['z_ms2'].values - grav_z

    a['lin_x'] = accel_lin_x[::step]
    a['lin_y'] = accel_lin_y[::step]
    a['lin_z'] = accel_lin_z[::step]

    fig, axes = plt.subplots(7, 1, figsize=(14, 20), sharex=True)

    # --- Raw (with gravity) ---
    axes[0].plot(a['time_s'], a['x_ms2'], linewidth=0.5, color='tab:red')
    axes[0].set_ylabel('X (m/s\u00b2)')
    axes[0].set_title('Raw accelerometer \u2014 lateral (X)')
    axes[0].grid(True, alpha=0.3)
    plot_event_markers(axes[0], events)

    axes[1].plot(a['time_s'], a['y_ms2'], linewidth=0.5, color='tab:green')
    axes[1].set_ylabel('Y (m/s\u00b2)')
    axes[1].set_title('Raw accelerometer \u2014 forward (Y)')
    axes[1].grid(True, alpha=0.3)
    plot_event_markers(axes[1], events)

    axes[2].plot(a['time_s'], a['z_ms2'], linewidth=0.5, color='tab:blue')
    axes[2].set_ylabel('Z (m/s\u00b2)')
    axes[2].set_title('Raw accelerometer \u2014 vertical (Z)')
    axes[2].grid(True, alpha=0.3)
    plot_event_markers(axes[2], events)

    # --- Gravity removed (linear acceleration) ---
    axes[3].plot(a['time_s'], a['lin_x'], linewidth=0.5, color='tab:red', alpha=0.8)
    axes[3].set_ylabel('X (m/s\u00b2)')
    axes[3].set_title('Linear acceleration \u2014 lateral (X), gravity removed')
    axes[3].grid(True, alpha=0.3)
    plot_event_markers(axes[3], events)

    axes[4].plot(a['time_s'], a['lin_y'], linewidth=0.5, color='tab:green', alpha=0.8)
    axes[4].set_ylabel('Y (m/s\u00b2)')
    axes[4].set_title('Linear acceleration \u2014 forward (Y), gravity removed')
    axes[4].grid(True, alpha=0.3)
    plot_event_markers(axes[4], events)

    axes[5].plot(a['time_s'], a['lin_z'], linewidth=0.5, color='tab:blue', alpha=0.8)
    axes[5].set_ylabel('Z (m/s\u00b2)')
    axes[5].set_title('Linear acceleration \u2014 vertical (Z), gravity removed')
    axes[5].grid(True, alpha=0.3)
    plot_event_markers(axes[5], events)

    # --- Magnitude of linear acceleration ---
    lin_mag = np.sqrt(a['lin_x']**2 + a['lin_y']**2 + a['lin_z']**2)
    axes[6].plot(a['time_s'], lin_mag, linewidth=0.5, color='tab:purple')
    axes[6].set_ylabel('|a| (m/s\u00b2)')
    axes[6].set_xlabel('Time (s)')
    axes[6].set_title('Linear acceleration magnitude (gravity removed)')
    axes[6].grid(True, alpha=0.3)
    plot_event_markers(axes[6], events)

    # Add legend for event markers
    if events is not None and len(events) > 0:
        from matplotlib.lines import Line2D
        legend_elements = [Line2D([0], [0], color='red', linestyle='--', label='pothole event'),
                           Line2D([0], [0], color='orange', linestyle='--', label='crack event')]
        axes[0].legend(handles=legend_elements, loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()

    # Store linear accel globally for use in GPS map and drill-down
    accel['lin_x'] = accel_lin_x
    accel['lin_y'] = accel_lin_y
    accel['lin_z'] = accel_lin_z
    accel['lin_mag'] = np.sqrt(accel_lin_x**2 + accel_lin_y**2 + accel_lin_z**2)
else:
    print('No accelerometer data.')

## Gyroscope — 3-axis time series

In [ ]:
if gyro is not None and len(gyro) > 0:
    step = max(1, len(gyro) // 10_000)
    g = gyro.iloc[::step]

    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

    axes[0].plot(g['time_s'], g['x_rads'], linewidth=0.5, color='tab:red')
    axes[0].set_ylabel('X (rad/s)')
    axes[0].set_title('Gyroscope (roll)')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(g['time_s'], g['y_rads'], linewidth=0.5, color='tab:green')
    axes[1].set_ylabel('Y (rad/s)')
    axes[1].set_title('Gyroscope (pitch)')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(g['time_s'], g['z_rads'], linewidth=0.5, color='tab:blue')
    axes[2].set_ylabel('Z (rad/s)')
    axes[2].set_xlabel('Time (s)')
    axes[2].set_title('Gyroscope (yaw)')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('No gyroscope data.')

## Speed profile (GPS)

In [ ]:
if gps is not None and len(gps) > 0 and 'speed_mps' in gps.columns:
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(gps['time_s'], gps['speed_mps'] * 3.6, linewidth=1, color='tab:orange')
    ax.set_ylabel('Speed (km/h)')
    ax.set_xlabel('Time (s)')
    ax.set_title('Vehicle speed')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()
else:
    print('No GPS speed data.')

## Spectrogram — Vertical acceleration (Z-axis)

Potholes produce broadband mid-frequency bursts (5-15 Hz).

In [ ]:
if accel is not None and len(accel) > 500:
    # Estimate actual sample rate from timestamps
    dt_ns = np.diff(accel['timestamp_ns'].values[:10000])
    fs = 1e9 / np.median(dt_ns)
    print(f'Estimated sample rate: {fs:.0f} Hz')

    # Compute spectrogram
    nperseg = min(512, len(accel) // 4)
    f, t, Sxx = signal.spectrogram(
        accel['z_ms2'].values, fs=fs, nperseg=nperseg,
        noverlap=nperseg // 2, scaling='density',
    )

    # Limit frequency range to 0-50 Hz (most interesting for potholes)
    f_max = 50
    f_mask = f <= f_max

    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.pcolormesh(
        t, f[f_mask], 10 * np.log10(Sxx[f_mask] + 1e-10),
        shading='gouraud', cmap='inferno',
    )
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_title('Z-axis acceleration spectrogram')
    plt.colorbar(im, ax=ax, label='Power (dB)')

    # Mark pothole frequency band
    ax.axhline(5, color='white', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axhline(15, color='white', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.text(t[0] + 0.5, 10, 'pothole band', color='white', fontsize=9, alpha=0.7)

    plt.tight_layout()
    plt.show()
else:
    print('Not enough accelerometer data for spectrogram.')

## GPS track map

Track colored by acceleration magnitude. Red = high acceleration spikes, green = calm driving.
Click the markers to see peak acceleration values.

In [ ]:
if (gps is not None and len(gps) > 1 and accel is not None and len(accel) > 0
        and 'lat_deg' in gps.columns):

    # Use linear acceleration magnitude (gravity removed) if available, else raw
    if 'lin_mag' in accel.columns:
        accel_mag = accel['lin_mag']
    else:
        accel_mag = np.sqrt(accel['x_ms2']**2 + accel['y_ms2']**2 + accel['z_ms2']**2)

    # For each GPS fix, find the max acceleration in a 1-second window around it
    gps_accel_max = []
    for _, row in gps.iterrows():
        t = row['time_s']
        mask = (accel['time_s'] >= t - 0.5) & (accel['time_s'] < t + 0.5)
        if mask.any():
            gps_accel_max.append(accel_mag[mask].max())
        else:
            gps_accel_max.append(0.0)
    gps_accel_max = np.array(gps_accel_max)

    # Normalize for color mapping
    vmin, vmax = 0.5, max(5.0, np.percentile(gps_accel_max, 98))
    norm = (gps_accel_max - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)

    def value_to_color(v):
        """Green (0) to Yellow (0.5) to Red (1)."""
        r = int(min(255, v * 2 * 255))
        g = int(min(255, (1 - v) * 2 * 255))
        return f'#{r:02x}{g:02x}00'

    # Create map centered on the track
    center_lat = gps['lat_deg'].mean()
    center_lon = gps['lon_deg'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=15)

    # Draw colored track segments
    for i in range(len(gps) - 1):
        color = value_to_color(norm[i])
        folium.PolyLine(
            locations=[
                [gps.iloc[i]['lat_deg'], gps.iloc[i]['lon_deg']],
                [gps.iloc[i + 1]['lat_deg'], gps.iloc[i + 1]['lon_deg']],
            ],
            color=color, weight=5, opacity=0.8,
        ).add_to(m)

    # Add markers at top-5 acceleration peaks
    top_indices = np.argsort(gps_accel_max)[-5:]
    for idx in top_indices:
        row = gps.iloc[idx]
        val = gps_accel_max[idx]
        folium.Marker(
            location=[row['lat_deg'], row['lon_deg']],
            popup=f"Peak: {val:.1f} m/s\u00b2 @ t={row['time_s']:.1f}s",
            icon=folium.Icon(color='red', icon='exclamation-sign'),
        ).add_to(m)

    # Start marker
    folium.Marker(
        location=[gps.iloc[0]['lat_deg'], gps.iloc[0]['lon_deg']],
        popup='Start',
        icon=folium.Icon(color='green', icon='play'),
    ).add_to(m)

    display(m)
else:
    print('No GPS data available for map.')

## Drill-down: zoom into a time window

Use `plot_window(t_start, t_end)` to zoom into a specific event.
Shows accelerometer (3-axis + magnitude), gyroscope, and speed on synced time axes.

In [ ]:
def plot_window(t_start_s, t_end_s):
    """Plot a zoomed time window with accel + gyro + speed overlaid."""
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

    # --- Raw Accelerometer ---
    if accel is not None:
        mask = (accel['time_s'] >= t_start_s) & (accel['time_s'] <= t_end_s)
        a = accel[mask]
        if len(a) > 0:
            axes[0].plot(a['time_s'], a['x_ms2'], linewidth=0.7, label='X', alpha=0.7)
            axes[0].plot(a['time_s'], a['y_ms2'], linewidth=0.7, label='Y', alpha=0.7)
            axes[0].plot(a['time_s'], a['z_ms2'], linewidth=0.7, label='Z', alpha=0.7)
    axes[0].set_ylabel('m/s\u00b2')
    axes[0].set_title(f'Raw accelerometer (with gravity) \u2014 {t_start_s:.1f}s \u2013 {t_end_s:.1f}s')
    axes[0].legend(loc='upper right', fontsize=8)
    axes[0].grid(True, alpha=0.3)
    plot_event_markers(axes[0], events, t_start_s, t_end_s)

    # --- Linear Accelerometer (gravity removed) + magnitude ---
    if accel is not None and 'lin_x' in accel.columns:
        mask = (accel['time_s'] >= t_start_s) & (accel['time_s'] <= t_end_s)
        a = accel[mask]
        if len(a) > 0:
            axes[1].plot(a['time_s'], a['lin_x'], linewidth=0.7, label='X', alpha=0.8)
            axes[1].plot(a['time_s'], a['lin_y'], linewidth=0.7, label='Y', alpha=0.8)
            axes[1].plot(a['time_s'], a['lin_z'], linewidth=0.7, label='Z', alpha=0.8)
            lin_mag = np.sqrt(a['lin_x']**2 + a['lin_y']**2 + a['lin_z']**2)
            axes[1].plot(a['time_s'], lin_mag, linewidth=1, label='|a|', color='black', alpha=0.4)
    axes[1].set_ylabel('m/s\u00b2')
    axes[1].set_title('Linear acceleration (gravity removed)')
    axes[1].legend(loc='upper right', fontsize=8)
    axes[1].grid(True, alpha=0.3)
    plot_event_markers(axes[1], events, t_start_s, t_end_s)

    # --- Gyroscope ---
    if gyro is not None:
        mask = (gyro['time_s'] >= t_start_s) & (gyro['time_s'] <= t_end_s)
        g = gyro[mask]
        if len(g) > 0:
            axes[2].plot(g['time_s'], g['x_rads'], linewidth=0.7, label='Roll', alpha=0.8)
            axes[2].plot(g['time_s'], g['y_rads'], linewidth=0.7, label='Pitch', alpha=0.8)
            axes[2].plot(g['time_s'], g['z_rads'], linewidth=0.7, label='Yaw', alpha=0.8)
    axes[2].set_ylabel('rad/s')
    axes[2].set_title('Gyroscope')
    axes[2].legend(loc='upper right', fontsize=8)
    axes[2].grid(True, alpha=0.3)
    plot_event_markers(axes[2], events, t_start_s, t_end_s)

    # --- Speed ---
    if gps is not None and 'speed_mps' in gps.columns:
        mask = (gps['time_s'] >= t_start_s) & (gps['time_s'] <= t_end_s)
        g_gps = gps[mask]
        if len(g_gps) > 0:
            axes[3].plot(g_gps['time_s'], g_gps['speed_mps'] * 3.6,
                         linewidth=1.5, color='tab:orange', marker='o', markersize=3)
    axes[3].set_ylabel('Speed (km/h)')
    axes[3].set_xlabel('Time (s)')
    axes[3].set_ylim(bottom=0)
    axes[3].grid(True, alpha=0.3)
    plot_event_markers(axes[3], events, t_start_s, t_end_s)

    plt.tight_layout()
    plt.show()


# Example: zoom into the first pothole event (at ~30s in synthetic data)
plot_window(28, 33)

In [ ]:
# Zoom into the second pothole event
plot_window(63, 67)

In [ ]:
# Zoom into the third pothole event
plot_window(93, 97)